# 从零实现 LoRA 与教学版 QLoRA：低秩适配、合并和可信发布

本 Notebook 只使用 PyTorch 基础张量与 nn.Module，手写 LoRALinear55、因果自注意力、TinyCausalLM55、groupwise 4-bit 冻结线性层和发布包装器。目标是把低秩增量从公式落实为可训练、可合并、可审计的代码，而不是调用 transformers、peft、bitsandbytes 或现成语言模型。

受控任务把基础映射 A→X、B→Y 适配为带领域标记后的 A→Y、B→X。它只证明实现能在微型闭集上学习指定规则，绝不代表对自然语言或未见领域具备泛化能力。

In [ ]:
import copy
import hashlib
import json
import math
import random
import warnings
from types import MappingProxyType

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED55 = 5501
random.seed(SEED55); np.random.seed(SEED55); torch.manual_seed(SEED55)
torch.set_num_threads(1)
DEVICE55 = torch.device("cpu")

def canonical_json55(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def sha55(raw):
    return hashlib.sha256(raw).hexdigest()

assert DEVICE55.type == "cpu"
assert torch.get_num_threads() == 1
assert torch.initial_seed() == SEED55

## 1. Token、数据与切分合同

每条序列形状为 ids:[T]，训练采用 next-token 目标 ids[1:]。基础数据与领域数据完全列入 DATA55，split 按记录 id 引用；发布摘要会绑定完整 tokenizer、全部记录、切分和训练 recipe，而不是只绑定一个模糊的数据版本字符串。

audit 记录是组合序列，用于验证已学习规则，不把闭集结果包装成真实泛化。生产中还应记录去重策略、许可、PII 处理、样本时间与上游快照。

In [ ]:
TOKENIZER55 = {
    "<pad>": 0, "<bos>": 1, "<eos>": 2,
    "A": 3, "B": 4, "X": 5, "Y": 6, "<domain>": 7,
}
ID_TO_TOKEN55 = {value: key for key, value in TOKENIZER55.items()}
DATA55 = [
    {"id": "base-a", "kind": "base", "tokens": [1, 3, 5, 2]},
    {"id": "base-b", "kind": "base", "tokens": [1, 4, 6, 2]},
    {"id": "adapt-a", "kind": "domain", "tokens": [1, 7, 3, 6, 2]},
    {"id": "adapt-b", "kind": "domain", "tokens": [1, 7, 4, 5, 2]},
    {"id": "audit-composed", "kind": "audit", "tokens": [1, 7, 3, 6, 4, 5, 2]},
]
SPLIT55 = {
    "base_train": ["base-a", "base-b"],
    "adapter_train": ["adapt-a", "adapt-b"],
    "audit": ["audit-composed"],
}

def records_by_ids55(names):
    table = {row["id"]: row for row in DATA55}
    return [table[name]["tokens"] for name in names]

def padded_batch55(sequences):
    width = max(map(len, sequences))
    result = torch.full((len(sequences), width), TOKENIZER55["<pad>"], dtype=torch.long)
    for row, sequence in enumerate(sequences):
        result[row, :len(sequence)] = torch.tensor(sequence)
    return result

base_batch55 = padded_batch55(records_by_ids55(SPLIT55["base_train"]))
adapt_batch55 = padded_batch55(records_by_ids55(SPLIT55["adapter_train"]))
assert base_batch55.shape == (2, 4)
assert adapt_batch55.shape == (2, 5)
assert set(ID_TO_TOKEN55) == set(TOKENIZER55.values())
assert len({row["id"] for row in DATA55}) == len(DATA55)
assert set(sum(SPLIT55.values(), [])) == {row["id"] for row in DATA55}
assert not (set(SPLIT55["base_train"]) & set(SPLIT55["adapter_train"]))

## 2. LoRA 数学、形状与初始化

对冻结权重 W:[out,in]，LoRA 不直接更新 W，而是学习 A:[r,in]、B:[out,r]：

y = x W^T + (alpha/r) x A^T B^T + b。

参数量从 out×in 变为 r(in+out)。这里 A 用小随机数初始化、B 初始化为零，所以第 0 步严格满足 ΔW=0，包装前后输出相同。第一步常只有 B 获得非零梯度；B 离开零点后，A 才会收到有效梯度，这是正常现象。

In [ ]:
class LoRALinear55(nn.Module):
    def __init__(self, base, rank=4, alpha=8.0):
        super().__init__()
        if not isinstance(base, nn.Linear) or rank < 1 or rank > min(base.in_features, base.out_features):
            raise ValueError("invalid_lora_base_or_rank")
        self.base = base
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scale = self.alpha / self.rank
        self.A = nn.Parameter(torch.empty(self.rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, self.rank))
        nn.init.normal_(self.A, std=0.02)
        self.register_buffer("_merge_state", torch.tensor(0, dtype=torch.uint8), persistent=True)
        self.register_buffer("_merged_A", torch.zeros_like(self.A), persistent=True)
        self.register_buffer("_merged_B", torch.zeros_like(self.B), persistent=True)
        self.register_buffer("_merged_delta", torch.zeros_like(self.base.weight), persistent=True)
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)

    @property
    def merged(self):
        return bool(int(self._merge_state.item()))

    def delta_weight(self):
        return (self.B @ self.A) * self.scale

    def _assert_state_consistent(self):
        state = int(self._merge_state.item())
        if state not in (0, 1):
            raise RuntimeError("invalid_persisted_merge_state")
        if state == 1 and self.training:
            raise RuntimeError("merged_adapter_in_training_mode")
        if state == 1 and (
            not torch.equal(self.A.detach(), self._merged_A)
            or not torch.equal(self.B.detach(), self._merged_B)
        ):
            raise RuntimeError("merged_adapter_parameters_changed")

    def forward(self, x):
        self._assert_state_consistent()
        base_value = self.base(x)
        if self.merged:
            return base_value
        return base_value + F.linear(x, self.delta_weight())

    def train(self, mode=True):
        if mode and self.merged:
            raise RuntimeError("merged_adapter_cannot_train")
        return super().train(mode)

    def state_dict(self, *args, **kwargs):
        self._assert_state_consistent()
        return super().state_dict(*args, **kwargs)

    @torch.no_grad()
    def merge(self):
        if self.training:
            raise RuntimeError("merge_requires_eval_mode")
        if self.merged:
            raise RuntimeError("adapter_already_merged")
        delta = self.delta_weight().detach()
        self._merged_A.copy_(self.A.detach())
        self._merged_B.copy_(self.B.detach())
        self._merged_delta.copy_(delta)
        self.base.weight.add_(delta)
        self._merge_state.fill_(1)

    @torch.no_grad()
    def unmerge(self):
        if not self.merged:
            raise RuntimeError("adapter_not_merged")
        self._assert_state_consistent()
        self.base.weight.sub_(self._merged_delta)
        self._merge_state.zero_()
        self._merged_A.zero_()
        self._merged_B.zero_()
        self._merged_delta.zero_()

probe_base55 = nn.Linear(6, 5)
probe_lora55 = LoRALinear55(copy.deepcopy(probe_base55), rank=2, alpha=4)
probe_x55 = torch.randn(7, 6)
assert torch.allclose(probe_lora55(probe_x55), probe_base55(probe_x55), atol=1e-7)
assert torch.count_nonzero(probe_lora55.delta_weight()).item() == 0
assert sum(p.numel() for p in (probe_lora55.A, probe_lora55.B)) == 22
assert all(not p.requires_grad for p in probe_lora55.base.parameters())
assert not probe_lora55.merged and int(probe_lora55._merge_state) == 0

## 3. 手写 Tiny causal attention 模型

输入 ids:[B,T] 经 token/position embedding 得到 x:[B,T,D]。注意力显式构造 q/k/v:[B,H,T,Dh]，分数除以 sqrt(Dh)，同时应用 causal mask 与 padding key mask。这里没有调用 nn.MultiheadAttention 或 Transformer。

复杂度仍是 O(B×H×T²×Dh)，LoRA 只减少可训练参数和优化器状态，不会自动消除注意力的二次复杂度。

In [ ]:
class CausalSelfAttention55(nn.Module):
    def __init__(self, dim=24, heads=4):
        super().__init__()
        if dim % heads:
            raise ValueError("dim_must_be_divisible_by_heads")
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x, valid_mask):
        if x.ndim != 3 or valid_mask.shape != x.shape[:2] or valid_mask.dtype != torch.bool:
            raise ValueError("invalid_attention_shapes")
        batch, length, _ = x.shape
        def split(linear):
            return linear(x).view(batch, length, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = split(self.q_proj), split(self.k_proj), split(self.v_proj)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        causal = torch.tril(torch.ones(length, length, dtype=torch.bool, device=x.device))
        allowed = causal[None, None] & valid_mask[:, None, None, :]
        scores = scores.masked_fill(~allowed, -torch.finfo(scores.dtype).max)
        weights = scores.softmax(-1)
        context = (weights @ v).transpose(1, 2).reshape(batch, length, self.dim)
        output = self.out_proj(context)
        return output * valid_mask.unsqueeze(-1), weights

class TinyCausalLM55(nn.Module):
    def __init__(self, vocab_size=8, dim=24, heads=4, max_length=12, pad_id=0):
        super().__init__()
        self.vocab_size, self.dim, self.heads, self.max_length, self.pad_id = vocab_size, dim, heads, max_length, pad_id
        if not 0 <= self.pad_id < self.vocab_size:
            raise ValueError("pad_id_out_of_range")
        self.token = nn.Embedding(vocab_size, dim)
        self.position = nn.Embedding(max_length, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.attention = CausalSelfAttention55(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim))
        self.head = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, input_ids):
        if input_ids.ndim != 2 or input_ids.dtype != torch.long:
            raise ValueError("input_ids_must_be_rank2_long")
        if input_ids.shape[1] < 1 or input_ids.shape[1] > self.max_length:
            raise ValueError("invalid_sequence_length")
        if input_ids.min() < 0 or input_ids.max() >= self.vocab_size:
            raise ValueError("token_id_out_of_range")
        valid = input_ids.ne(self.pad_id)
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)
        x = self.token(input_ids) + self.position(positions)[None]
        attended, weights = self.attention(self.norm1(x), valid)
        x = x + attended
        x = x + self.ff(self.norm2(x)) * valid.unsqueeze(-1)
        return self.head(x), weights

shape_model55 = TinyCausalLM55()
shape_logits55, shape_weights55 = shape_model55(adapt_batch55)
assert shape_logits55.shape == (2, 5, len(TOKENIZER55))
assert shape_weights55.shape == (2, 4, 5, 5)
assert torch.allclose(torch.triu(shape_weights55[0, 0], diagonal=1), torch.zeros(5, 5), atol=1e-7)

## 4. 基础训练与标准 LoRA 领域适配

基础模型先拟合两条基础序列。领域适配时冻结 embedding、attention、FFN 和原始输出头，只替换输出头为 LoRALinear55。adapter loss 只监督 X/Y 位置，避免把微型示例中的 BOS、EOS 频率当成业务目标。

真实训练应报告不同 rank、target modules、学习率、数据量与随机种子的消融；本例的闭集准确率只是正确性探针。

In [ ]:
def token_loss55(model, batch, supervised_ids=None):
    logits, _ = model(batch)
    targets = batch[:, 1:]
    valid = targets.ne(TOKENIZER55["<pad>"])
    if supervised_ids is not None:
        selected = torch.zeros_like(valid)
        for token_id in supervised_ids:
            selected |= targets.eq(token_id)
        valid &= selected
    if not valid.any():
        raise ValueError("empty_supervision")
    losses = F.cross_entropy(logits[:, :-1].reshape(-1, model.vocab_size), targets.reshape(-1), reduction="none")
    return losses.view_as(targets)[valid].mean()

base_model55 = TinyCausalLM55()
base_optimizer55 = torch.optim.AdamW(base_model55.parameters(), lr=0.035, weight_decay=0.0)
base_start55 = float(token_loss55(base_model55, base_batch55))
for _ in range(35):
    base_optimizer55.zero_grad()
    loss55 = token_loss55(base_model55, base_batch55)
    loss55.backward(); base_optimizer55.step()
base_end55 = float(token_loss55(base_model55, base_batch55))
assert base_end55 < base_start55 * 0.12

lora_model55 = copy.deepcopy(base_model55)
lora_model55.head = LoRALinear55(lora_model55.head, rank=4, alpha=8)
for name55, parameter55 in lora_model55.named_parameters():
    parameter55.requires_grad_(name55 in {"head.A", "head.B"})
trainable_names55 = [name for name, p in lora_model55.named_parameters() if p.requires_grad]
assert trainable_names55 == ["head.A", "head.B"]
lora_initial55 = float(token_loss55(lora_model55, adapt_batch55, {5, 6}))
optimizer55 = torch.optim.AdamW([lora_model55.head.A, lora_model55.head.B], lr=0.09, weight_decay=0.0)
first_b_grad55 = second_a_grad55 = 0.0
for step55 in range(50):
    optimizer55.zero_grad()
    adapter_loss55 = token_loss55(lora_model55, adapt_batch55, {5, 6})
    adapter_loss55.backward()
    if step55 == 0:
        first_b_grad55 = float(lora_model55.head.B.grad.abs().sum())
        assert torch.count_nonzero(lora_model55.head.A.grad).item() == 0
    if step55 == 1:
        second_a_grad55 = float(lora_model55.head.A.grad.abs().sum())
    optimizer55.step()
lora_final55 = float(token_loss55(lora_model55, adapt_batch55, {5, 6}))
assert lora_final55 < lora_initial55 * 0.03
assert first_b_grad55 > 0 and second_a_grad55 > 0
assert all(p.grad is None for name, p in lora_model55.named_parameters() if name not in trainable_names55)

## 5. 持久化 merge 状态、checkpoint round-trip 与 fail-closed unmerge

标准 LoRA 部署可把 delta-W 加回浮点 W，消除额外的小矩阵乘。但 merged 不是普通 Python bool：base 已经含增量，若 checkpoint 丢失该状态，加载后 forward 会再加一次 adapter。这里把状态、merge 时的 A/B 快照与实际 delta 都注册为持久 buffer。

merge 只允许在 eval 状态执行；merged 模块拒绝切回 train。forward、state_dict 与 unmerge 都核对 A/B 是否仍等于 merge 快照。若调用者在 merged 状态修改 adapter，系统拒绝继续或反合并，而不是用新 delta 去减旧 base，造成不可逆污染。该 checkpoint 是“可恢复的 merged LoRA 状态”，不是可以脱离 base 解释的 adapter-only 文件。

In [ ]:
merge_probe55 = copy.deepcopy(lora_model55.head).eval()
hidden_probe55 = torch.randn(3, 4, base_model55.dim)
unmerged_value55 = merge_probe55(hidden_probe55)
merge_probe55.merge()
merged_value55 = merge_probe55(hidden_probe55)
assert merge_probe55.merged and int(merge_probe55._merge_state) == 1
assert torch.allclose(unmerged_value55, merged_value55, atol=2e-5, rtol=2e-5)

merged_checkpoint55 = copy.deepcopy(merge_probe55.state_dict())
restored_merged55 = LoRALinear55(
    nn.Linear(base_model55.dim, len(TOKENIZER55), bias=False), rank=4, alpha=8
)
restored_merged55.load_state_dict(merged_checkpoint55, strict=True)
try:
    restored_merged55(hidden_probe55)
    raise AssertionError("loaded merged checkpoint ran in training mode")
except RuntimeError as error55:
    assert str(error55) == "merged_adapter_in_training_mode"
restored_merged55.eval()
assert restored_merged55.merged
assert torch.allclose(restored_merged55(hidden_probe55), merged_value55, atol=0, rtol=0)
try:
    restored_merged55.train()
    raise AssertionError("merged checkpoint entered training mode")
except RuntimeError as error55:
    assert str(error55) == "merged_adapter_cannot_train"

merge_probe55.unmerge()
assert not merge_probe55.merged and int(merge_probe55._merge_state) == 0
assert torch.allclose(unmerged_value55, merge_probe55(hidden_probe55), atol=2e-5, rtol=2e-5)
try:
    merge_probe55.unmerge()
    raise AssertionError("unmerged adapter was unmerged twice")
except RuntimeError as error55:
    assert str(error55) == "adapter_not_merged"

changed_after_merge55 = copy.deepcopy(lora_model55.head).eval()
changed_after_merge55.merge()
with torch.no_grad():
    changed_after_merge55.A[0, 0].add_(0.25)
try:
    changed_after_merge55.unmerge()
    raise AssertionError("changed merged adapter was subtracted from base")
except RuntimeError as error55:
    assert str(error55) == "merged_adapter_parameters_changed"
try:
    changed_after_merge55.state_dict()
    raise AssertionError("inconsistent merged checkpoint was saved")
except RuntimeError as error55:
    assert str(error55) == "merged_adapter_parameters_changed"

def domain_predictions55(model):
    prompts = torch.tensor([[1, 7, 3], [1, 7, 4]], dtype=torch.long)
    logits, _ = model(prompts)
    return logits[:, -1].argmax(-1).tolist()

assert domain_predictions55(lora_model55) == [TOKENIZER55["Y"], TOKENIZER55["X"]]

## 6. 教学版 groupwise int4 frozen base + LoRA

每个输出行按连续 group 求 scale=max(abs(w))/7，再把权重舍入到 [-7,7] 的 int8 容器；逻辑上每项只需 4 bit，但这里没有做 nibble packing。forward 时显式反量化，base 始终冻结，只有 A/B 更新。

这不是正式 QLoRA：原论文使用适合近似正态权重的 NF4、double quantization 和 paged optimizer；本教学版是对称均匀量化，scale 仍保存为 float32，也没有显存分页。名称和边界必须说清，不能把“4-bit 容器示意”冒充 bitsandbytes 的内核与内存收益。

In [ ]:
class GroupwiseInt4FrozenLinear55(nn.Module):
    def __init__(self, linear, group_size=8):
        super().__init__()
        if not isinstance(linear, nn.Linear) or group_size < 1:
            raise ValueError("invalid_quantized_linear")
        if not torch.isfinite(linear.weight).all() or (linear.bias is not None and not torch.isfinite(linear.bias).all()):
            raise ValueError("nonfinite_weight_cannot_be_quantized")
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.group_size = int(group_size)
        groups = math.ceil(self.in_features / self.group_size)
        padded = groups * self.group_size
        weight = F.pad(linear.weight.detach().float(), (0, padded - self.in_features))
        grouped = weight.view(self.out_features, groups, self.group_size)
        scale = grouped.abs().amax(-1).clamp_min(1e-8) / 7.0
        qweight = torch.round(grouped / scale.unsqueeze(-1)).clamp(-7, 7).to(torch.int8)
        self.register_buffer("qweight", qweight)
        self.register_buffer("scale", scale)
        if linear.bias is None:
            self.register_buffer("bias", None)
        else:
            self.register_buffer("bias", linear.bias.detach().float().clone())

    def dequantize(self):
        dense = (self.qweight.float() * self.scale.unsqueeze(-1)).reshape(self.out_features, -1)
        return dense[:, :self.in_features]

    def forward(self, x):
        if x.shape[-1] != self.in_features:
            raise ValueError("quantized_linear_input_mismatch")
        return F.linear(x, self.dequantize(), self.bias)

class QuantizedLoRALinear55(nn.Module):
    def __init__(self, base, rank=4, alpha=8.0):
        super().__init__()
        if not isinstance(base, GroupwiseInt4FrozenLinear55):
            raise TypeError("base_must_be_groupwise_int4")
        if rank < 1 or rank > min(base.in_features, base.out_features):
            raise ValueError("invalid_quantized_lora_rank")
        self.base, self.rank, self.alpha = base, int(rank), float(alpha)
        self.scale_factor = self.alpha / self.rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.normal_(self.A, std=0.02)

    def forward(self, x):
        return self.base(x) + F.linear(x, (self.B @ self.A) * self.scale_factor)

quant_base55 = GroupwiseInt4FrozenLinear55(base_model55.head, group_size=8)
dense_weight55 = base_model55.head.weight.detach()
relative_error55 = (quant_base55.dequantize() - dense_weight55).norm() / dense_weight55.norm().clamp_min(1e-8)
assert quant_base55.qweight.dtype == torch.int8
assert quant_base55.qweight.min() >= -7 and quant_base55.qweight.max() <= 7
assert 0 < relative_error55 < 0.25
padded_dense55 = F.pad(dense_weight55, (0, quant_base55.qweight.shape[-1] * quant_base55.qweight.shape[1] - dense_weight55.shape[1]))
grouped_dense55 = padded_dense55.view_as(quant_base55.qweight)
grouped_dequant55 = quant_base55.qweight.float() * quant_base55.scale.unsqueeze(-1)
assert torch.all((grouped_dense55 - grouped_dequant55).abs() <= quant_base55.scale.unsqueeze(-1) / 2 + 1e-7)
bad_quant_linear55 = nn.Linear(3, 2)
with torch.no_grad(): bad_quant_linear55.weight[0, 0] = float("nan")
try:
    GroupwiseInt4FrozenLinear55(bad_quant_linear55, group_size=2)
    raise AssertionError("nonfinite base weight was quantized")
except ValueError as error55:
    assert str(error55) == "nonfinite_weight_cannot_be_quantized"
quant_adapter_probe55 = QuantizedLoRALinear55(quant_base55, rank=4, alpha=8)
quant_input55 = torch.randn(4, base_model55.dim)
assert torch.allclose(quant_adapter_probe55(quant_input55), quant_base55(quant_input55), atol=1e-7)

## 7. 在量化冻结头上训练 adapter

qlora_model55 保留同一个手写 causal attention 主体，只把浮点输出头换成量化冻结 base + 低秩增量。训练过程中 qweight/scale 是 buffer，不进入优化器；所有其他参数也显式冻结。

正式系统还需要量化 kernel、设备布局、混合精度、梯度累积和 OOM 回退测试。本例反量化后再做 F.linear，强调的是数值语义，不承诺真实 4-bit 推理加速。

In [ ]:
qlora_model55 = copy.deepcopy(base_model55)
qlora_model55.head = QuantizedLoRALinear55(
    GroupwiseInt4FrozenLinear55(base_model55.head, group_size=8), rank=4, alpha=8
)
for name55, parameter55 in qlora_model55.named_parameters():
    parameter55.requires_grad_(name55 in {"head.A", "head.B"})
q_trainable55 = [(name, p.numel()) for name, p in qlora_model55.named_parameters() if p.requires_grad]
assert [name for name, _ in q_trainable55] == ["head.A", "head.B"]
assert sum(count for _, count in q_trainable55) == 4 * (base_model55.dim + len(TOKENIZER55))

q_start55 = float(token_loss55(qlora_model55, adapt_batch55, {5, 6}))
q_optimizer55 = torch.optim.AdamW([qlora_model55.head.A, qlora_model55.head.B], lr=0.10, weight_decay=0.0)
q_first_b_grad55 = q_second_a_grad55 = 0.0
for q_step55 in range(40):
    q_optimizer55.zero_grad()
    q_loss55 = token_loss55(qlora_model55, adapt_batch55, {5, 6})
    q_loss55.backward()
    if q_step55 == 0:
        q_first_b_grad55 = float(qlora_model55.head.B.grad.abs().sum())
        assert torch.count_nonzero(qlora_model55.head.A.grad).item() == 0
    if q_step55 == 1:
        q_second_a_grad55 = float(qlora_model55.head.A.grad.abs().sum())
    q_optimizer55.step()
q_end55 = float(token_loss55(qlora_model55, adapt_batch55, {5, 6}))
assert q_end55 < q_start55 * 0.04
assert domain_predictions55(qlora_model55) == [TOKENIZER55["Y"], TOKENIZER55["X"]]
assert q_first_b_grad55 > 0 and q_second_a_grad55 > 0
assert all(p.grad is None for name, p in qlora_model55.named_parameters() if name not in {"head.A", "head.B"})

roundtrip_state55 = copy.deepcopy(qlora_model55.state_dict())
roundtrip_model55 = copy.deepcopy(qlora_model55)
roundtrip_model55.load_state_dict(roundtrip_state55)
assert torch.equal(roundtrip_model55.head.base.qweight, qlora_model55.head.base.qweight)
assert torch.allclose(roundtrip_model55(adapt_batch55)[0], qlora_model55(adapt_batch55)[0], atol=0, rtol=0)

## 8. 可信发布：信任根必须在包外

state 摘要逐项绑定参数 key、dtype、shape 与连续 bytes。metadata 绑定完整 tokenizer、DATA55、SPLIT55、模型 config、量化/训练 recipe 和允许主体。包内 self_digest 只能发现传输损坏；攻击者若能换模型，也能重算它，所以真正信任根是进程外配置注入后暴露为 MappingProxyType 的 TRUSTED_RELEASES55。

loader 不返回裸 nn.Module，而返回 PublishedLoRA55；包装器固定领域 prompt、主体、tokenizer 和最大长度，减少调用方绕过发布语义的机会。

In [ ]:
def state_digest55(state):
    digest = hashlib.sha256()
    for key in sorted(state):
        tensor = state[key].detach().cpu().contiguous()
        header = canonical_json55({
            "key": key, "dtype": str(tensor.dtype), "shape": list(tensor.shape)
        }).encode("utf-8")
        raw = tensor.numpy().tobytes()
        digest.update(len(header).to_bytes(8, "big")); digest.update(header)
        digest.update(len(raw).to_bytes(8, "big")); digest.update(raw)
    return digest.hexdigest()

CONFIG55 = {"vocab_size": 8, "dim": 24, "heads": 4, "max_length": 12, "pad_id": 0, "rank": 4, "alpha": 8.0, "group_size": 8}
RECIPE55 = {
    "algorithm": "educational_uniform_groupwise_int4_plus_lora",
    "seed": SEED55, "optimizer": "AdamW", "base_steps": 35,
    "standard_lora_steps": 50, "adapter_lr": 0.10, "steps": 40,
    "supervised_token_ids": [5, 6], "quant_range": [-7, 7],
    "not_nf4": True, "not_double_quant": True, "packed_nibbles": False,
}

def release_digest55(package):
    envelope = {
        "release_id": package["release_id"],
        "metadata": package["metadata"],
        "state_digest": state_digest55(package["state"]),
    }
    return sha55(canonical_json55(envelope).encode("utf-8"))

def build_release55(model):
    state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    package = {
        "release_id": "lora-domain-v1",
        "metadata": {
            "config": copy.deepcopy(CONFIG55),
            "tokenizer": copy.deepcopy(TOKENIZER55),
            "data": copy.deepcopy(DATA55),
            "split": copy.deepcopy(SPLIT55),
            "recipe": copy.deepcopy(RECIPE55),
            "allowed_subject": "team-search",
        },
        "state": state,
        "state_digest": state_digest55(state),
    }
    package["self_digest"] = release_digest55(package)
    return package

RELEASE_PACKAGE55 = build_release55(qlora_model55)
TRUSTED_RELEASES55 = MappingProxyType({"lora-domain-v1": release_digest55(RELEASE_PACKAGE55)})

def validate_metadata55(metadata):
    tokenizer = metadata["tokenizer"]
    if sorted(tokenizer.values()) != list(range(len(tokenizer))) or tokenizer.get("<pad>") != 0:
        raise ValueError("invalid_tokenizer_contract")
    records = metadata["data"]; split = metadata["split"]
    ids = [row["id"] for row in records]
    referenced = sum(split.values(), [])
    if len(ids) != len(set(ids)) or sorted(ids) != sorted(referenced) or len(referenced) != len(set(referenced)):
        raise ValueError("invalid_data_split_contract")
    if any(any(not isinstance(token, int) or token not in tokenizer.values() for token in row["tokens"]) for row in records):
        raise ValueError("invalid_bound_training_tokens")
    recipe = metadata["recipe"]
    if not recipe.get("not_nf4") or recipe.get("packed_nibbles"):
        raise ValueError("quantization_semantics_mismatch")
    if metadata["config"]["vocab_size"] != len(tokenizer) or metadata["config"]["pad_id"] != tokenizer["<pad>"]:
        raise ValueError("config_tokenizer_mismatch")

class PublishedLoRA55:
    def __init__(self, model, metadata, subject):
        self._model = model.eval()
        self.tokenizer = MappingProxyType(copy.deepcopy(metadata["tokenizer"]))
        self.id_to_token = MappingProxyType({value: key for key, value in self.tokenizer.items()})
        self.subject = subject
        self.max_length = metadata["config"]["max_length"]

    @torch.no_grad()
    def predict_domain(self, symbol):
        if self.subject != "team-search":
            raise PermissionError("subject_not_authorized")
        if symbol not in ("A", "B"):
            raise ValueError("domain_symbol_must_be_A_or_B")
        prompt = torch.tensor([[self.tokenizer["<bos>"], self.tokenizer["<domain>"], self.tokenizer[symbol]]])
        if prompt.shape[1] > self.max_length:
            raise ValueError("prompt_too_long")
        logits, _ = self._model(prompt)
        return self.id_to_token[int(logits[0, -1].argmax())]

def load_published55(package, subject):
    release_id = package.get("release_id")
    actual = release_digest55(package)
    if release_id not in TRUSTED_RELEASES55 or actual != TRUSTED_RELEASES55[release_id]:
        raise PermissionError("untrusted_release_digest")
    if package.get("self_digest") != actual or package.get("state_digest") != state_digest55(package["state"]):
        raise ValueError("corrupt_release")
    validate_metadata55(package["metadata"])
    if subject != package["metadata"]["allowed_subject"]:
        raise PermissionError("subject_not_authorized")
    config = package["metadata"]["config"]
    model = TinyCausalLM55(config["vocab_size"], config["dim"], config["heads"], config["max_length"], config["pad_id"])
    model.head = QuantizedLoRALinear55(
        GroupwiseInt4FrozenLinear55(model.head, config["group_size"]), config["rank"], config["alpha"]
    )
    model.load_state_dict(package["state"], strict=True)
    return PublishedLoRA55(model, package["metadata"], subject)

published55 = load_published55(copy.deepcopy(RELEASE_PACKAGE55), "team-search")
assert published55.predict_domain("A") == "Y"
assert published55.predict_domain("B") == "X"
assert isinstance(TRUSTED_RELEASES55, MappingProxyType)
assert RELEASE_PACKAGE55["state_digest"] == state_digest55(RELEASE_PACKAGE55["state"])

## 9. 失败测试、保存语义与生产差距

merge 状态机先拒绝训练、双重合并、adapter 漂移后的 unmerge 与不一致保存。下列测试再区分“包内校验”和“可信发布”：攻击者修改 adapter 后同时更新 state_digest 与 self_digest，内部字段完全自洽，但包外 registry 仍保存原发布摘要，因此必须拒绝。还测试未知 token、错误主体和非法 rank。

生产部署还要解决多 adapter 路由、并发 merge 锁、base model 精确版本、量化 kernel ABI、显存峰值、回滚和在线质量门禁。LoRA 文件不能脱离 base 权重与 tokenizer 单独解释。

In [ ]:
forged55 = copy.deepcopy(RELEASE_PACKAGE55)
forged55["state"]["head.B"][0, 0] += 1.0
forged55["state_digest"] = state_digest55(forged55["state"])
forged55["self_digest"] = release_digest55(forged55)
assert forged55["self_digest"] == release_digest55(forged55)
try:
    load_published55(forged55, "team-search")
    raise AssertionError("fully resigned forged package was trusted")
except PermissionError as error55:
    assert str(error55) == "untrusted_release_digest"

try:
    load_published55(RELEASE_PACKAGE55, "other-team")
    raise AssertionError("unauthorized subject was accepted")
except PermissionError as error55:
    assert str(error55) == "subject_not_authorized"
try:
    published55.predict_domain("C")
    raise AssertionError("unknown symbol was accepted")
except ValueError as error55:
    assert str(error55) == "domain_symbol_must_be_A_or_B"
try:
    LoRALinear55(nn.Linear(2, 2), rank=3)
    raise AssertionError("rank larger than matrix dimensions was accepted")
except ValueError as error55:
    assert str(error55) == "invalid_lora_base_or_rank"

## 10. 结论与原始资料

本例完成了四条闭环：低秩增量的初始化/梯度，标准 LoRA 的 merge/unmerge，教学版 groupwise int4 frozen base + adapter，以及带外信任根的发布加载。最重要的边界是：参数高效不等于计算免费，教学量化不等于 NF4 内核，闭集适配成功不等于真实泛化，自签哈希也不等于可信发布。

原始资料：

- LoRA: Low-Rank Adaptation of Large Language Models：https://arxiv.org/abs/2106.09685
- QLoRA: Efficient Finetuning of Quantized LLMs：https://arxiv.org/abs/2305.14314
- PyTorch nn.Module 官方文档：https://pytorch.org/docs/stable/generated/torch.nn.Module.html